# General Instructions

- Before executing the training loop, we must download the required open-source packages.
- We utilize the standard transformers library for loading the model and tokenizer, and datasets for fetching the text-to-SQL schema data.

In [1]:
!pip install -q transformers datasets accelerate torch TQDM

- We configure global hyperparameters, system paths, and setup the CUDA execution environment.
- Gemma-3 models benefit significantly from mixed-precision execution (bfloat16 or float16) to save memory and match tensor-core calculation layouts.

In [29]:
import torch
import random
import numpy as np
import os
import gc

# Force Python garbage collection
gc.collect()

# Free up all unallocated cached memory blocks inside the PyTorch CUDA allocator
torch.cuda.empty_cache()

# Define global runtime parameters
MODEL_ID = "google/gemma-3-270m"
DATASET_ID = "SuperMax991/spider-text2sql"
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
EPOCHS = 3
MAX_LENGTH = 1024

# LoRA Specific Parameters
LORA_R = 8              # Rank of low-rank matrix decomposition
LORA_ALPHA = 16         # Scaling factor
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"] # Modules to adapt

# Seed for reproducibility
def seed_everything(seed: int = 42) -> None:
    """Sets random seeds across numpy, random, and torch for execution consistency."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using runtime device platform target: {device}")

Using runtime device platform target: cuda


- LoRA (Low-Rank Adaptation) acts upon frozen linear layers.
- For an original weight matrix W_0 \in \mathbb{R}^{d \times k}, we decompose its weight update \Delta W into two low-rank matrices B \in \mathbb{R}^{d \times r} and A \in \mathbb{R}^{r \times k}, where r << min(d, k).
- The forward pass computes: h = W_0 \cdot x + \frac{\alpha}{r} (B \cdot A \cdot x)
- Matrix A is initialized via a Gaussian distribution, and matrix B is initialized to zeros.
- This ensures that at step 0, \Delta W = 0, and the model behaves exactly like the base model.

In [13]:
import torch.nn as nn
import torch.nn.functional as F
class LoraLinear(nn.Module):
    """
    Custom implementation of a Low-Rank Adaptation (LoRA) Linear Layer from scratch.
    Ensures absolute dtype and device synchronization with the wrapped base model.
    """
    def __init__(
        self,
        base_layer: nn.Linear,
        r: int = 8,
        lora_alpha: float = 16.0,
        lora_dropout: float = 0.05
    ):
        super().__init__()
        self.base_layer = base_layer
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        # FIX: Extract the exact dtype and device from the existing base layer weights
        target_dtype = base_layer.weight.dtype
        target_device = base_layer.weight.device

        # Instantiate adapter parameters matching the base layer's type environments exactly
        self.lora_A = nn.Parameter(
            torch.empty((r, self.in_features), dtype=target_dtype, device=target_device)
        )
        self.lora_B = nn.Parameter(
            torch.empty((self.out_features, r), dtype=target_dtype, device=target_device)
        )

        self.dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()

        self.reset_parameters()
        self.freeze_base_weights()

    def reset_parameters(self) -> None:
        """Initializes low-rank matrices using Kaiming-uniform for A and Zeros for B."""
        # Note: kaiming_uniform_ handles alternative float precision types natively
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def freeze_base_weights(self) -> None:
        """Freezes gradients on the wrapped base neural parameters."""
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Computes forward pass through frozen layer combined with scaled low-rank adapter track.
        """
        base_output = self.base_layer(x)

        # Both x and lora_A/lora_B are now guaranteed to share the exact same float precision context
        lora_output = F.linear(self.dropout(x), self.lora_A)
        lora_output = F.linear(lora_output, self.lora_B)

        return base_output + (lora_output * self.scaling)

- Since we are not using the `peft` framework, we must manually parse the model's module graph.
- We use PyTorch's native `model.named_modules()` generator. This flat iterator handles the tree traversal under the hood, returning every module and nested submodule along with its string path.
- We split the module's path to find its immediate parent container. This allows us to swap the target layer out for our custom `LoraLinear` layer in-place using Python's `setattr`.

In [4]:
import torch.nn as nn

def apply_lora_to_model_iterative(
    model: nn.Module,
    target_modules: list[str],
    r: int,
    alpha: float
) -> None:
    """
    Iteratively scans the model architecture graph using a flat loop, locating layers
    matching the target_modules list, and replaces them in-place with the custom LoRA wrapper.
    """
    # Create a static list of named modules to avoid iterating over the graph
    # while mutating it structural layouts in real-time.
    module_items = list(model.named_modules())

    for name, module in module_items:
        # Check if the current module is a linear layer and matches our target names (e.g., 'q_proj')
        if isinstance(module, nn.Linear) and any(name.endswith(target) for target in target_modules):

            # Split the hierarchical path to isolate the parent container and the layer attribute name
            # Example: 'model.layers.0.self_attn.q_proj' -> ['model.layers.0.self_attn', 'q_proj']
            parts = name.split('.')
            parent_path = ".".join(parts[:-1])
            layer_name = parts[-1]

            # Retrieve the parent module instance using the model context
            parent_module = model.get_submodule(parent_path) if parent_path else model

            # Initialize our custom LoRA wrapper layer around the existing weights
            lora_layer = LoraLinear(base_layer=module, r=r, lora_alpha=alpha)

            # Dynamically swap the base layer with the custom LoRA adapter layer
            setattr(parent_module, layer_name, lora_layer)
            print(f"Successfully adapted linear target: {name}")

def print_trainable_parameters(model: nn.Module) -> None:
    """Calculates and displays frozen vs trainable parameter sizes."""
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"Trainable Parameters: {trainable_params:,} | "
        f"Total Parameters: {all_param:,} | "
        f"Trainable Ratio: {100 * trainable_params / all_param:.4f}%"
    )

- Instead of relying on Hugging Face's dataset mapping functions, we implement a custom PyTorch `Dataset` subclass (`TextToSqlDataset`).
- This class wraps the raw text data and defines the `__getitem__` method to retrieve individual examples.
- Tokenization is isolated here so that raw strings are dynamically prepared for processing.
- We also compute the masking bounds here to set prompt tokens to `-100`, ensuring our model only learns from calculating error margins on target SQL tokens.

In [30]:
from torch.utils.data import Dataset, Subset
from transformers import AutoTokenizer
from datasets import load_dataset

class TextToSqlDataset(Dataset):
    """
    Custom PyTorch Dataset for formatting and tokenizing Text-to-SQL tasks.
    Prepares text sequences and establishes cross-entropy target sequence masking labels.
    """
    def __init__(self, raw_dataset, tokenizer: AutoTokenizer, max_length: int = 512):
        self.dataset = raw_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.dataset)

    def _format_prompt(self, db_schema: str, question: str, query: str) -> tuple[str, str]:
        """Constructs text components for contextual generation."""
        prompt = f"### Context Schema:\n{db_schema}\n\n### Question:\n{question}\n\n### SQL:\n"
        full_text = prompt + query
        return prompt, full_text

    def __getitem__(self, idx: int) -> dict[str, list[int]]:
        # Extract row properties from raw dataset item
        item = self.dataset[idx]
        schema = item['db_schema']
        question = item['question']
        sql = item['query']

        prompt, full_text = self._format_prompt(schema, question, sql)

        # Tokenize both pieces individually to determine token lengths accurately
        tokenized_prompt = self.tokenizer(prompt, truncation=True, max_length=self.max_length, padding=False)
        tokenized_full = self.tokenizer(full_text, truncation=True, max_length=self.max_length, padding=False)

        input_ids = tokenized_full['input_ids']
        attention_mask = tokenized_full['attention_mask']

        prompt_len = len(tokenized_prompt['input_ids'])

        # Build the Target Labels array:
        # Pad the prompt tokens section with -100 so PyTorch CrossEntropyLoss ignores them.
        labels = [-100] * prompt_len + input_ids[prompt_len:]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

print("Initializing Tokenizer and Downloading Raw Data...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the raw dataset without using Hugging Face processing methods
raw_spider_dataset = load_dataset(DATASET_ID)

# Instantiate the custom PyTorch Dataset configurations
full_train_dataset = TextToSqlDataset(raw_spider_dataset['train'], tokenizer, max_length=MAX_LENGTH)

# 2. Use PyTorch Subset to extract exactly the first 1,000 samples
indices = list(range(1000))
train_dataset = Subset(full_train_dataset, indices)

print(f"Custom PyTorch dataset initialized.")
print(f"Original Dataset Size: {len(full_train_dataset)} | Sliced Training Subset Size: {len(train_dataset)}")

Initializing Tokenizer and Downloading Raw Data...
Custom PyTorch dataset initialized.
Original Dataset Size: 7000 | Sliced Training Subset Size: 1000


- When using variable-length sequence data, PyTorch's default data loader cannot batch rows automatically because tensors must have identical dimensions.
- To solve this, we write a standalone `collate_fn`. This function takes a list of variable-length dictionaries from our `TextToSqlDataset`, pads them to the maximum sequence length found *within that specific batch*, and stacks them into uniform PyTorch tensors.
- This dynamic padding optimization keeps sequences compact, reducing VRAM usage and accelerating execution compared to fixed global padding constraints.

In [31]:
import torch
import torch.nn as nn

def text2sql_collate_fn(batch: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    """
    Collates a list of samples into a single dynamically padded mini-batch tensor structure.

    Args:
        batch: List of dictionaries containing individual sequence token lists.

    Returns:
        A dict containing padded input_ids, attention_masks, and token label tensors.
    """
    pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    label_pad_id = -100

    # Extract lists of sequences from structural batch input arguments
    batch_input_ids = [torch.tensor(item['input_ids'], dtype=torch.long) for item in batch]
    batch_attention_mask = [torch.tensor(item['attention_mask'], dtype=torch.long) for item in batch]
    batch_labels = [torch.tensor(item['labels'], dtype=torch.long) for item in batch]

    # Dynamically pad each tensor array grouping to match the longest sequence in this current batch
    padded_input_ids = nn.utils.rnn.pad_sequence(
        batch_input_ids, batch_first=True, padding_value=pad_token_id
    )
    padded_attention_mask = nn.utils.rnn.pad_sequence(
        batch_attention_mask, batch_first=True, padding_value=0
    )
    padded_labels = nn.utils.rnn.pad_sequence(
        batch_labels, batch_first=True, padding_value=label_pad_id
    )

    return {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "labels": padded_labels
    }

# Connect the dataset and custom collate function to a standard PyTorch DataLoader configuration
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=text2sql_collate_fn,
    drop_last=True # Avoids partial uneven batches at end of epoch splits
)

print(f"PyTorch DataLoader configured with {len(train_loader)} iteration steps per training cycle.")

PyTorch DataLoader configured with 1000 iteration steps per training cycle.


- Now we load the `Gemma-3-270m` base model.
- Once loaded, we verify its internal parameter layouts, inject our custom LoRA adapters,and lock the base parameters to prevent updating.

In [14]:
from transformers import AutoModelForCausalLM
import math

print(f"Loading base language model framework targets: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
)

print("\n--- Step 1: Global Model Freeze ---")
# CRITICAL STEP: Explicitly freeze every single parameter across the entire base architecture.
# This targets embeddings, layer norms, and attention heads that LoRA won't touch.
for param in model.parameters():
    param.requires_grad = False

print("All base model parameters successfully frozen.")
print_trainable_parameters(model) # Should show exactly 0% trainable parameters

print("\n--- Step 2: Injecting Custom LoRA Adapters ---")
# Apply our iterative function. This swaps target layers with LoraLinear wrappers.
# Inside LoraLinear, lora_A and lora_B are instantiated as fresh nn.Parameters,
# which inherently default to requires_grad=True.
apply_lora_to_model_iterative(model, TARGET_MODULES, r=LORA_R, alpha=LORA_ALPHA)

print("\n--- Final Parameter Verification ---")
# This will now display ONLY the parameters of your custom lora_A and lora_B weights as trainable.
print_trainable_parameters(model)

# Relocate the uniquely configured hybrid model to the GPU
model = model.to(device)

Loading base language model framework targets: google/gemma-3-270m


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]


--- Step 1: Global Model Freeze ---
All base model parameters successfully frozen.
Trainable Parameters: 0 | Total Parameters: 268,098,176 | Trainable Ratio: 0.0000%

--- Step 2: Injecting Custom LoRA Adapters ---
Successfully adapted linear target: model.layers.0.self_attn.q_proj
Successfully adapted linear target: model.layers.0.self_attn.k_proj
Successfully adapted linear target: model.layers.0.self_attn.v_proj
Successfully adapted linear target: model.layers.0.self_attn.o_proj
Successfully adapted linear target: model.layers.1.self_attn.q_proj
Successfully adapted linear target: model.layers.1.self_attn.k_proj
Successfully adapted linear target: model.layers.1.self_attn.v_proj
Successfully adapted linear target: model.layers.1.self_attn.o_proj
Successfully adapted linear target: model.layers.2.self_attn.q_proj
Successfully adapted linear target: model.layers.2.self_attn.k_proj
Successfully adapted linear target: model.layers.2.self_attn.v_proj
Successfully adapted linear target: m

- This cell executes the model optimization loop.
- We configure a standard AdamW optimizer. Because the base model layers are frozen,
- AdamW only updates the weight values inside the custom LoRA adapter matrices (`lora_A` and `lora_B`).
- We also include gradient accumulation to simulate larger effective batches using less VRAM.

In [32]:
from tqdm import tqdm

# Filter optimizer target allocations to parameters that require gradients
trainable_weights = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_weights, lr=LEARNING_RATE)

print("Beginning fine-tuning training loop operations...")
model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS}")

    optimizer.zero_grad()

    for step, batch in progress_bar:
        # Move items to runtime processing device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass evaluation
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        # Scale loss to adjust for gradient accumulation steps
        loss = loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward()

        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        # Step optimizer parameters based on accumulation frequency configurations
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            # Clip gradients to prevent gradient explosions
            torch.nn.utils.clip_grad_norm_(trainable_weights, max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        progress_bar.set_postfix({"batch_loss": loss.item() * GRADIENT_ACCUMULATION_STEPS})

    average_epoch_loss = epoch_loss / len(train_loader)
    print(f"Completed Epoch {epoch+1} Evaluation. Metrics - Average Training Loss Value: {average_epoch_loss:.5f}")

Beginning fine-tuning training loop operations...


Epoch 1/3: 100%|██████████| 1000/1000 [03:55<00:00,  4.25it/s, batch_loss=0.413]


Completed Epoch 1 Evaluation. Metrics - Average Training Loss Value: 0.74627


Epoch 2/3: 100%|██████████| 1000/1000 [03:57<00:00,  4.21it/s, batch_loss=0.474]


Completed Epoch 2 Evaluation. Metrics - Average Training Loss Value: 0.42663


Epoch 3/3: 100%|██████████| 1000/1000 [03:54<00:00,  4.26it/s, batch_loss=0.176]

Completed Epoch 3 Evaluation. Metrics - Average Training Loss Value: 0.30700


- Finally, we test the fine-tuned model's ability to generate SQL queries.
- We switch the model to evaluation mode (`model.eval()`) to disable dropout, format an unseen test sample, and use standard autoregressive generation loop steps to print the predicted SQL query.

In [47]:
def format_text_to_sql_prompt(schema: str, question: str, sql: str = "") -> dict:
    """Constructs prompt structure for instructional causal model tasks."""
    prompt = f"### Context Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
    full_text = prompt + sql if sql else prompt
    return {"prompt": prompt, "full_text": full_text}

In [48]:
def generate_sql_inference(schema: str, question: str) -> str:
    """Uses the fine-tuned model to generate an SQL query from an evaluation prompt."""
    model.eval()
    formatted = format_text_to_sql_prompt(schema, question)

    inputs = tokenizer(formatted["prompt"], return_tensors="pt").to(device)

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False, # Deterministic greedy decoding for programmatic reliability
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode full token stream array back to plain text representation
    full_generation_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Isolate newly generated content trailing downstream from instruction prompt boundaries
    generated_sql = full_generation_text[len(formatted["prompt"]):].strip()
    return generated_sql

# Example test data point evaluation execution
test_schema = "Table: users (user_id INT, user_name VARCHAR, join_date DATE); Table: purchases (purchase_id INT, user_id INT, amount FLOAT)"
test_question = "Find the total amount spent by user_name 'Alice'"

print("Executing inference sample test...")
predicted_sql = generate_sql_inference(test_schema, test_question)
print(f"\nPredicted SQL Query Result:\n{predicted_sql}")

Executing inference sample test...

Predicted SQL Query Result:
SELECT sum(amount) FROM purchases WHERE user_id  =  'Alice';

### SQL:
SELECT sum(amount) FROM table EXCEPT SELECT sum(amount) FROM table EXCEPT SELECT count(*) FROM purchase GROUP BY user_id HAVING sum(amount)  >=  1000000;


- To avoid wasting storage space by saving the entire 270M parameter base model, we extract and save only the custom weights we trained.
- This results in a small file (a few megabytes) containing just the trained LoRA adapter parameters.

In [34]:
def extract_and_save_lora_weights(model: nn.Module, save_path: str) -> None:
    """
    Scans the model parameter tree, isolates custom trained lora module parameters,
    and saves only those parameters to a local file.
    """
    lora_state_dict = {}
    for name, param in model.named_parameters():
        if "lora_" in name:
            lora_state_dict[name] = param.cpu()

    torch.save(lora_state_dict, save_path)
    print(f"Successfully saved custom LoRA state dictionary to: {save_path} ({len(lora_state_dict)} tensors written)")

extract_and_save_lora_weights(model, "custom_gemma3_text2sql_lora.pt")

Successfully saved custom LoRA state dictionary to: custom_gemma3_text2sql_lora.pt (144 tensors written)


- Since we implemented LoRA from scratch, our adapters cannot be saved via standard
model.save_pretrained() shortcuts.
- Instead, we bundle our custom weights tensor file
along with a metadata card into a local directory, and use the `huggingface_hub`
repository API to push our lean adapter artifacts straight to the cloud.

In [45]:
import torch
import torch.nn as nn
from huggingface_hub import HfApi
from google.colab import userdata
import os

print("--- Step 1: Merging LoRA Weights into Base Model ---")

def merge_and_uncollapse_lora(model: nn.Module) -> None:
    """
    Finds all custom LoraLinear layers, computes their delta weights, merges them
    into the base weights in-place, and replaces the layer with a standard nn.Linear module.
    """
    module_items = list(model.named_modules())

    for name, module in module_items:
        if isinstance(module, LoraLinear):
            # 1. Access the underlying base linear layer
            base_layer = module.base_layer

            # 2. Calculate delta weights: Delta W = (B @ A) * scaling
            # lora_B shape: [OutFeatures, r], lora_A shape: [r, InFeatures]
            # Matrix multiplication results in: [OutFeatures, InFeatures]
            with torch.no_grad():
                delta_weight = torch.matmul(module.lora_B, module.lora_A) * module.scaling

                # 3. Add the delta weights directly to the frozen base weight matrix
                base_layer.weight.data += delta_weight.to(base_layer.weight.dtype)

            # 4. Find the parent module to swap the custom layer back to a standard nn.Linear
            parts = name.split('.')
            parent_path = ".".join(parts[:-1])
            layer_name = parts[-1]
            parent_module = model.get_submodule(parent_path) if parent_path else model

            # 5. Replace our custom wrapper with the newly updated, pure standard linear layer
            setattr(parent_module, layer_name, base_layer)
            print(f"Successfully merged weights and restored standard linear layer for: {name}")

# Execute the weight merge on our trained model
merge_and_uncollapse_lora(model)

print("\n--- Step 2: Authenticating and Preparing Full Model Upload ---")
# Get Hugging Face write token from Colab secrets
hf_token_writes = userdata.get("HF_TOKEN_WRITES")
if hf_token_writes is None:
    raise ValueError("Hugging Face write token (HF_TOKEN_WRITES) not found in Colab secrets. Please add it to access the Hugging Face Hub.")

# Initialize HfApi with the token
hf_api = HfApi(token=hf_token_writes)

HF_USERNAME = userdata.get("HF_USERNAME") # Replace with your actual HF account username
if HF_USERNAME is None:
    raise ValueError("Hugging Face username (HF_USERNAME) not found in Colab secrets. Please add it.")

REPO_NAME = "gemma3-270m-spider-text2sql-lora-merged"
FULL_REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
LOCAL_SAVE_DIR = "/content/merged_lora_gemma3_text2sql"

print(f"Saving merged full model and tokenizer configurations locally...")

# Create the local directory if it doesn't exist
os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)

# Save the unified weights and internal structure configuration files
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

print(f"\n--- Step 3: Executing Multi-Asset Upload via Raw HfApi ---")
# Create the target repository block on the hub if it doesn't already exist
print(f"Ensuring remote repository exists at: {FULL_REPO_ID}")
hf_api.create_repo(
    repo_id=FULL_REPO_ID,
    repo_type="model",
    exist_ok=True
)

# Upload the entire local directory structure natively
print(f"Uploading files from '{LOCAL_SAVE_DIR}' to Hugging Face Hub...")
hf_api.upload_folder(
    folder_path=LOCAL_SAVE_DIR,
    repo_id=FULL_REPO_ID,
    repo_type="model"
)

print(f"\n[SUCCESS] The standalone model and card are fully live!")
print(f"Your repository can be viewed publicly at: https://huggingface.co/{FULL_REPO_ID}")

--- Step 1: Merging LoRA Weights into Base Model ---

--- Step 2: Authenticating and Preparing Full Model Upload ---
Saving merged full model and tokenizer configurations locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Step 3: Executing Multi-Asset Upload via Raw HfApi ---
Ensuring remote repository exists at: AyuK007/gemma3-270m-spider-text2sql-lora-merged
Uploading files from '/content/merged_lora_gemma3_text2sql' to Hugging Face Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3_text2sql/tokenizer.json:  23%|##3       | 7.69MB / 33.4MB            

  ...ext2sql/model.safetensors:  13%|#3        | 72.0MB /  536MB            


[SUCCESS] The standalone model and card are fully live!
Your repository can be viewed publicly at: https://huggingface.co/AyuK007/gemma3-270m-spider-text2sql-lora-merged
